# Análise Exploratória de Dados (AED)

Este notebook foi criado para organizar a análise exploratória dos dados do projeto. Os objetivos são:  (1) Entender estrutura e qualidade dos dados; (2) Identificar padrões iniciais e possíveis outliers; e (3) Preparar insights para etapas posteriores do projeto.

### Introdução

A Aventura é uma produtora e gestora cultural referência no Brasil, com forte atuação na criação de espetáculos e na gestão de teatros, promovendo experiências artísticas de alta qualidade para diferentes públicos. Inserido nesse contexto, o projeto propõe o desenvolvimento de uma solução digital para o Portal Aventura, com foco em ampliar o alcance de seus conteúdos e fortalecer a conexão entre espectadores e o ecossistema cultural da empresa.

### Contextualização do Problema

Atualmente, o Portal Aventura não possui um sistema estruturado de recomendação de conteúdos e espetáculos. Dessa forma, a navegação do usuário ocorre principalmente a partir do acesso a páginas específicas, sem a apresecntação de sugestões relacionadas a outras experiências disponíveis no portal, o que pode reduzir o potencial de conversão e geração de receita.

### Objetivo do Sistema

O objetivo do sistema é desenvolver uma solução de recomendação inteligente capaz de sugerir espetáculos, conteúdos e experiências alinhadas aos interesses dos usuários. Com isso, busca-se aumentar o tempo de permanência no portal, estimular a navegação entre diferentes conteúdos e impulsionar a venda de ingressos, criando uma jornada mais conectada e personalizada dentro do ecossistema da Aventura.

# Importação dos dados e tratamento

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

sns.set_theme(style="whitegrid")

base_path = Path("../database_sympla.csv")

if not base_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado em: {base_path}")

df = pd.read_csv(base_path)

# Cópia para transformações de análise
df_eda = df.copy()

# Tratando vírgula/ponto e possíveis símbolos
df_eda["VALOR_NUM"] = (
    df_eda["VALOR DO INGRESSO"].astype(str)
    .str.replace(r"[^0-9,.-]", "", regex=True)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
)
df_eda["VALOR_NUM"] = pd.to_numeric(df_eda["VALOR_NUM"], errors="coerce")

# Conversão de datas
df_eda["DATA_APRESENTACAO_DT"] = pd.to_datetime(df_eda["DATA DA APRESENTAÇÃO"], errors="coerce")
df_eda["DATA_COMPRA_DT"] = pd.to_datetime(df_eda["DATA DA COMPRA"], errors="coerce")

# Antecedência de compra em dias (compra -> apresentação), arredondada para inteiro
df_eda["ANTECEDENCIA_DIAS"] = (
    (df_eda["DATA_APRESENTACAO_DT"] - df_eda["DATA_COMPRA_DT"]).dt.total_seconds() / 86400
).round().astype("Int64")

print(f"Linhas: {df_eda.shape[0]} | Colunas: {df_eda.shape[1]}")
df_eda.head()

## Criação do dataset para sistema de recomendação

In [ ]:
# CRIAÇÃO DO DATASET PARA CF (FILTRAGEM COLABORATIVA)

# Selecionar colunas necessárias
df_rec = df_eda[
    [
        "ID COMPRADOR.1.ID USUÁRIO",
        "NOME ESPETÁCULO",
        "VALOR_NUM",
        "TIPO"
    ]
].copy()

# Remover nulos essenciais
df_rec = df_rec.dropna(subset=["ID COMPRADOR.1.ID USUÁRIO", "NOME ESPETÁCULO"])

# Padronizar texto
df_rec["NOME ESPETÁCULO"] = df_rec["NOME ESPETÁCULO"].astype(str).str.strip()
df_rec["TIPO"] = df_rec["TIPO"].astype(str).str.strip()


# AGRUPAMENTO
dataset = (
    df_rec
    .groupby(["ID COMPRADOR.1.ID USUÁRIO", "NOME ESPETÁCULO"])
    .agg(
        ranking=("NOME ESPETÁCULO", "size"),
        valor_medio_ingresso=("VALOR_NUM", "mean"),
        tipo_evento=("TIPO", "first")
    )
    .reset_index()
)


# PADRONIZAÇÃO FINAL
dataset = dataset.rename(columns={
    "ID COMPRADOR.1.ID USUÁRIO": "user_id",
    "NOME ESPETÁCULO": "item_id"
})

dataset = dataset[
    ["user_id", "item_id", "ranking", "valor_medio_ingresso", "tipo_evento"]
]


# Salvar CSV
output_path = Path("../dataset/dataset.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

dataset.to_csv(output_path, index=False)

print("Dataset salvo em:", output_path)
dataset.head()

## Carregamento do dataset final

In [ ]:
df_dataset = pd.read_csv("../dataset/dataset.csv")
df_dataset.head()

## 1) Descrição do dataset final de recomendação

Nesta etapa respondemos:
- Quantas linhas e colunas existem?
- O que cada variável representa?
- Existem dados faltantes?
- Há registros duplicados?

In [ ]:
print(f"Dimensão do dataset final: {df_dataset.shape[0]:,} linhas x {df_dataset.shape[1]} colunas".replace(",", "."))

dicionario = pd.DataFrame(
    {
        "Variável": [
            "user_id",
            "item_id",
            "ranking",
            "valor_medio_ingresso",
            "tipo_evento",
        ],
        "Descrição": [
            "Identificador único do usuário/comprador.",
            "Identificador do item recomendado, representado pelo nome do espetáculo.",
            "Quantidade de compras do usuário para aquele espetáculo, usada como feedback implícito.",
            "Valor médio pago pelo usuário naquele espetáculo.",
            "Categoria/tipo do evento.",
        ],
    }
)

dicionario

In [ ]:
faltantes = pd.DataFrame(
    {
        "qtd_faltante": df_dataset.isna().sum(),
        "perc_faltante": (df_dataset.isna().mean() * 100).round(4),
    }
).sort_values("qtd_faltante", ascending=False)

print("Total de linhas duplicadas:", df_dataset.duplicated().sum())
faltantes

## 2) Estatísticas descritivas do dataset final

In [ ]:
df_dataset.describe(include="all")

## 3) Análise de Balanceamento e Diagnóstico da Matriz

O sucesso de um modelo de **Filtragem Colaborativa** reside na qualidade das interações entre usuários e itens. Nesta seção, realizamos um diagnóstico técnico baseado em três pilares:

1.  **Esparsidade da Matriz:** Calculamos o percentual de células vazias na matriz usuário-item. Uma esparsidade muito alta (ex: >99%) exige algoritmos de decomposição de matriz (como SVD).
2.  **Distribuição de Engajamento:** Identificamos se a base é dominada por usuários "hiperativos" ou institucionais, o que pode enviesar o aprendizado (bias).
3.  **Popularidade do Catálogo:** Avaliamos se o consumo está concentrado em poucos espetáculos (Hits) ou se há uma distribuição saudável para itens de nicho (Long Tail).

In [ ]:
# Métricas de Matriz
n_users = df_dataset["user_id"].nunique()
n_items = df_dataset["item_id"].nunique()
n_ratings = len(df_dataset)

# Cálculo de Esparsidade
rating_matrix_size = n_users * n_items
sparsity = (1 - n_ratings / rating_matrix_size) * 100

print(f"--- Métricas de Matriz Usuário-Item ---")
print(f"Número de usuários únicos: {n_users}")
print(f"Número de itens únicos: {n_items}")
print(f"Total de interações: {n_ratings}")
print(f"Esparsidade da matriz: {sparsity:.4f}%")

# Verificação de desbalanceamento de ranking
print("\n--- Proporção de Ranking (Volume de Recompras) ---")
print(df_dataset["ranking"].value_counts(normalize=True).map('{:.2%}'.format))

### 3.1) Diagnóstico da Matriz Usuário-Item

A **esparsidade** calculada acima indica o percentual de interações "vazias" que o nosso modelo precisará prever. Em datasets de entretenimento e cultura, uma alta esparsidade é comum. Isso justifica o uso futuro de modelos de **Fatoração de Matrizes (SVD)** ou **Filtragem Colaborativa baseada em Vizinhos**, que são eficazes em encontrar padrões mesmo quando os usuários possuem poucas interações documentadas.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df_dataset, x="ranking")
plt.title("Distribuição dos rankings (Original)")
plt.xlabel("Ranking - quantidade de compras")
plt.ylabel("Quantidade de relações usuário-item")
plt.show()

plt.figure(figsize=(10, 4))
df_dataset["user_id"].value_counts().head(20).plot(kind="bar", color='indianred')
plt.title("Top 20 usuários - Identificação de Outlier Extremo")
plt.xlabel("Usuário")
plt.ylabel("Quantidade de itens")
plt.tight_layout()
plt.show()

### 3.2) Detalhamento da Estratégia de Balanceamento: Trimagem por Percentis

Para garantir a eficácia do sistema de recomendação da **Aventura**, aplicamos uma estratégia de balanceamento focada na integridade da matriz usuário-item através da **Trimagem baseada em Percentis (Quantile-based Trimming)**.

#### 1. O Problema: Outliers de Volume
Em sistemas de recomendação baseados em filtragem colaborativa, o desbalanceamento mais crítico ocorre quando há uma disparidade extrema no volume de interações por usuário. 
* **O Risco:** Usuários "hiperativos" (com centenas de compras) funcionam como ruído. Eles "puxam" os cálculos de similaridade para si, fazendo com que o modelo recomende apenas o que esse super-usuário consumiu, ignorando os padrões reais da maioria orgânica.
* **A Causa:** Geralmente, esses perfis representam contas institucionais, bots de coleta ou pontos de venda física, e não o comportamento de um espectador individual.

#### 2. A Técnica: Trimagem por Percentil 99,9%
Em vez de definir um limite fixo e arbitrário, utilizamos a estatística descritiva para encontrar o ponto de corte natural do dataset.

**Como a técnica funciona:**
1. **Ordenação:** Calculamos a frequência de consumo para cada ID de usuário.
2. **Cálculo do Percentil:** Estabelecemos o **percentil 99,9%** como o limite superior de comportamento "orgânico". 
3. **Ponto de Corte:** Este cálculo define que 99,9% dos usuários da base consumiram até um determinado número de itens. Qualquer valor acima disso é estatisticamente tratado como uma anomalia (outlier).
4. **Normalização:** Os usuários que compõem o 0,1% restante da pirâmide de consumo são removidos, limpando o sinal que será enviado ao modelo.

#### 3. Preservação da Cauda Longa (Long Tail)
O balanceamento não visa apenas remover o topo, mas proteger a base. A estratégia garante que, ao limpar os usuários hiperativos, não prejudiquemos a diversidade de itens.

* **Cabeça do Catálogo (Head):** Itens populares (hits) que o modelo aprende facilmente.
* **Cauda do Catálogo (Tail):** Espetáculos de nicho que possuem poucas avaliações.

A estratégia adotada permite que o modelo mantenha a **capacidade de descoberta (Discovery)**, pois ao remover o ruído dos super-usuários, os padrões de consumo dos usuários comuns sobre itens da "cauda" tornam-se mais visíveis e fáceis de aprender pelo algoritmo.

#### 4. Benefícios para o Modelo de Recomendação
* **Estabilização da Variância:** Reduz o erro médio ao evitar distorções nas distâncias vetoriais.
* **Personalização Real:** As recomendações passam a ser baseadas em comportamentos de pessoas reais.
* **Mitigação do Viés de Popularidade:** Impede que o sistema se torne um "top 10" viciado, permitindo recomendações mais assertivas e variadas.

In [ ]:
# Definindo o limite de corte (Percentil 99.9)
limite_corte = df_dataset["user_id"].value_counts().quantile(0.999)

# Identificando usuários acima do limite
contagem_user = df_dataset["user_id"].value_counts()
usuarios_remover = contagem_user[contagem_user > limite_corte].index

# Criando o dataset balanceado
df_dataset_bal = df_dataset[~df_dataset["user_id"].isin(usuarios_remover)].copy()

# Salvando o dataset balanceado para o CSV final
df_dataset_bal.to_csv("../dataset/dataset.csv", index=False)

print(f"Usuários removidos (outliers): {list(usuarios_remover)}")
print(f"Volume original: {len(df_dataset)} | Volume pós-balanceamento: {len(df_dataset_bal)}")

In [ ]:
plt.figure(figsize=(10, 4))
df_dataset_bal["item_id"].value_counts().head(20).plot(kind="bar", color='seagreen')
plt.title("Top 20 itens por quantidade de usuários (Pós-Balanceamento)")
plt.xlabel("Item")
plt.ylabel("Quantidade de usuários")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 3.3) Validação: Curva de Cauda Longa (Long Tail)

Após a normalização, analisamos a distribuição do catálogo. Esperamos uma curva onde a "Cabeça" (itens hits) não abafe a capacidade do modelo de sugerir itens da "Cauda" (nicho), cumprindo a função de Discovery do portal Aventura.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Densidade Logarítmica
user_counts = df_dataset_bal.groupby("user_id").size()
sns.histplot(user_counts, bins=30, kde=True, ax=axes[0], color="purple")
axes[0].set_yscale('log')
axes[0].set_title("Densidade de Interações por Usuário (Escala Log)")

# Gráfico 2: Curva de Cauda Longa
item_pop = df_dataset_bal.groupby("item_id").size().sort_values(ascending=False).reset_index()
item_pop.columns = ['item_id', 'count']

plt.figure(figsize=(12, 5))
plt.fill_between(range(len(item_pop)), item_pop['count'], color="skyblue", alpha=0.4)
plt.plot(range(len(item_pop)), item_pop['count'], color="Slateblue", alpha=0.8, linewidth=2)
plt.axvline(x=len(item_pop)*0.2, color='red', linestyle='--', label='Top 20% (Cabeça)')
plt.title("Análise de Cauda Longa - Balanceamento de Itens")
plt.legend()
plt.show()

### 3.4) Conclusão da Qualidade do Dataset

Após a aplicação das técnicas de engenharia de dados, o dataset apresenta as seguintes garantias de qualidade:

* **Robustez Estatística:** A remoção de outliers de volume reduziu o ruído, estabilizando a variância das interações.
* **Integridade da Informação:** O uso de feedback implícito (ranking baseado em volume de compra) foi normalizado, mantendo a escala de relevância entre os usuários.
* **Prontidão para Modelagem:** A base resultante, agora armazenada em `dataset.csv`, possui o equilíbrio ideal entre usuários ativos e diversidade de itens, estando apta para a fase de treinamento dos algoritmos de recomendação.

## 4) Análise complementar da base original

As análises abaixo utilizam a base original para complementar o entendimento dos dados, considerando variáveis que podem apoiar uma futura abordagem híbrida de recomendação.

### 4.1 Distribuição de preço

In [ ]:
# Resumo estatístico do preço (unidade em R$)
resumo_preco = (
    df_eda["VALOR_NUM"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_frame("Preço (R$)")
    .round(2)
)
resumo_preco

### 4.2 Composição por tipo, teatro e canal de venda

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Distribuição de valores
sns.histplot(df_eda["VALOR_NUM"].dropna(), bins=40, kde=True, ax=axes[0, 0], color="#4C78A8")
axes[0, 0].set_title("Distribuição do valor do ingresso")
axes[0, 0].set_xlabel("Valor do ingresso")

# Top tipos
top_tipos = df_eda["TIPO"].value_counts().head(8)
sns.barplot(x=top_tipos.values, y=top_tipos.index, ax=axes[0, 1], color="#59A14F")
axes[0, 1].set_title("Top 8 tipos de evento")
axes[0, 1].set_xlabel("Quantidade de ingressos")
axes[0, 1].set_ylabel("Tipo")

# Top teatros
top_teatros = df_eda["TEATRO"].value_counts().head(8)
sns.barplot(x=top_teatros.values, y=top_teatros.index, ax=axes[1, 0], color="#F28E2B")
axes[1, 0].set_title("Top teatros por volume")
axes[1, 0].set_xlabel("Quantidade de ingressos")
axes[1, 0].set_ylabel("Teatro")

# Canal de venda
canal_counts = df_eda["CANAL DA VENDA"].value_counts()
sns.barplot(x=canal_counts.values, y=canal_counts.index, ax=axes[1, 1], color="#E15759")
axes[1, 1].set_title("Distribuição por canal de venda")
axes[1, 1].set_xlabel("Quantidade de ingressos")
axes[1, 1].set_ylabel("Canal")

plt.tight_layout()
plt.show()

## 5) Análise bivariada complementar

Comportamento de compra e diferença de preço entre categorias.

In [ ]:
# Preço por tipo (foco nos tipos com maior volume)
tipos_top = df_eda["TIPO"].value_counts().head(6).index
base_tipos = df_eda[df_eda["TIPO"].isin(tipos_top)].copy()

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=base_tipos,
    x="TIPO",
    y="VALOR_NUM",
    order=tipos_top,
    showfliers=False,
    palette="Set2",
)
plt.xticks(rotation=25, ha="right")
plt.title("Distribuição de preço por tipo de evento (sem outliers)")
plt.xlabel("Tipo")
plt.ylabel("Valor do ingresso")
plt.tight_layout()
plt.show()

base_tipos.groupby("TIPO")["VALOR_NUM"].agg(["count", "median", "mean"]).sort_values("median", ascending=False)

In [ ]:
# Relação entre canal de venda e método de pagamento (proporção por canal)
ct = pd.crosstab(df_eda["CANAL DA VENDA"], df_eda["MÉTODO DE PAGAMENTO"], normalize="index")

# Seleciona os métodos mais frequentes para visualização legível
metodos_top = df_eda["MÉTODO DE PAGAMENTO"].value_counts().head(8).index
ct_plot = ct[ct.columns.intersection(metodos_top)]

plt.figure(figsize=(12, 5))
sns.heatmap(ct_plot, annot=True, fmt=".2f", cmap="Blues")
plt.title("Proporção dos métodos de pagamento por canal de venda")
plt.xlabel("Método de pagamento")
plt.ylabel("Canal da venda")
plt.tight_layout()
plt.show()

In [ ]:
# Antecedência de compra vs valor do ingresso (amostra para performance visual)
base_relacao = df_eda[["ANTECEDENCIA_DIAS", "VALOR_NUM"]].dropna().copy()
base_relacao = base_relacao[(base_relacao["ANTECEDENCIA_DIAS"] >= 0) & (base_relacao["ANTECEDENCIA_DIAS"] <= 180)]

amostra = base_relacao.sample(min(12000, len(base_relacao)), random_state=42)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=amostra, x="ANTECEDENCIA_DIAS", y="VALOR_NUM", alpha=0.25, s=16)
sns.regplot(data=amostra, x="ANTECEDENCIA_DIAS", y="VALOR_NUM", scatter=False, color="red")
plt.title("Relação entre antecedência de compra e valor do ingresso")
plt.xlabel("Antecedência (dias)")
plt.ylabel("Valor do ingresso")
plt.tight_layout()
plt.show()

amostra[["ANTECEDENCIA_DIAS", "VALOR_NUM"]].corr()

## 6) Sequenciamento e priorização da entrega

A entrega foi organizada em etapas sequenciais. Primeiro, foi realizada a importação e limpeza da base original. Em seguida, foi criado o dataset final para recomendação, estruturado com `user_id`, `item_id`, `ranking`, `valor_medio_ingresso` e `tipo_evento`.

Após a geração do CSV, o dataset foi carregado novamente no notebook para validação, descrição estatística e análise de balanceamento. Por fim, foram realizadas análises complementares da base original para apoiar possíveis evoluções futuras do sistema de recomendação.

A prioridade principal foi garantir a estrutura mínima exigida para filtragem colaborativa, com usuário, item e ranking. As variáveis adicionais foram mantidas como apoio para análises futuras e possíveis abordagens híbridas.

## 7) Conclusão

O dataset final foi estruturado para alimentar um sistema de recomendação, contendo `user_id`, `item_id`, `ranking`, `valor_medio_ingresso` e `tipo_evento`.

Como a base original não possui avaliações explícitas dos usuários, o `ranking` foi construído como feedback implícito, representando a quantidade de compras de cada usuário para cada espetáculo.

A análise exploratória permitiu verificar a estrutura, a qualidade e o balanceamento do dataset. As variáveis complementares, como valor médio do ingresso e tipo do evento, podem apoiar evoluções futuras para uma abordagem híbrida de recomendação.